# PrimeTrade.ai Hiring Assignment
## Bitcoin Fear & Greed vs Hyperliquid Trader Performance

### Objectives
- Merge sentiment and trading datasets
- Analyze profitability by sentiment
- Compute win rates
- Statistical testing
- Trader segmentation
- Actionable insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from sklearn.cluster import KMeans

fg = pd.read_csv('fear_greed_index.csv')
tr = pd.read_csv('historical_data.csv')

fg['date']=pd.to_datetime(fg['date'])
tr['date']=pd.to_datetime(tr['Timestamp'],unit='ms').dt.date
tr['date']=pd.to_datetime(tr['date'])

merged = tr.merge(fg[['date','classification']],on='date',how='left')
merged.head()


In [ ]:
# Sentiment Distribution
merged['classification'].value_counts().plot(kind='bar')
plt.title('Trades by Sentiment')
plt.show()


In [ ]:
# Average PnL by Sentiment
merged.groupby('classification')['Closed PnL'].mean().sort_values().plot(kind='bar')
plt.title('Average Closed PnL')
plt.show()


In [ ]:
# Win Rate
merged['win']=(merged['Closed PnL']>0).astype(int)
win_rate=merged.groupby('classification')['win'].mean()*100
print(win_rate)


In [ ]:
# T-Test Fear vs Greed
fear=merged.loc[merged['classification']=='Fear','Closed PnL'].dropna()
greed=merged.loc[merged['classification']=='Greed','Closed PnL'].dropna()
print(ttest_ind(fear,greed,equal_var=False))


In [ ]:
# Top Traders
top=merged.groupby('Account')['Closed PnL'].sum().sort_values(ascending=False).head(10)
print(top)


In [ ]:
# Trader Clustering
features=merged.groupby('Account').agg({
    'Closed PnL':'mean',
    'Size USD':'mean',
    'Fee':'mean'
}).dropna()

kmeans=KMeans(n_clusters=3, random_state=42, n_init=10)
features['cluster']=kmeans.fit_predict(features)
features.head()
